# AFM 423 — GPR2: LASSO & Ridge Regression
**Authors:** Hugh Jiang, Nolan Zhang, Vivian Guo  
**Course:** AFM 423 W26 — Topics in Financial Econometrics  

This notebook implements the LASSO and Ridge Regression models for the GPR2 project.  
It produces all outputs consumed by `Eval_Metrics.ipynb` for model evaluation.

---
## Pipeline Overview
```
STEP 0  Install & load packages
STEP 1  Load data_ml.RData → inspect structure
STEP 2  Preprocessing  → winsorise, cross-sectional standardise, lag
STEP 3  Rolling-window loop → LASSO predictions + Ridge predictions
STEP 4  Collect OOS predictions & weights into data frames
STEP 5  Save outputs  → sourced directly by Eval_Metrics.ipynb
```
**Everything in STEP 5 is what Eval_Metrics.ipynb expects as inputs.**

---
## STEP 0 — Install & Load Packages

In [ ]:
# Install if needed (comment out after first run)
# install.packages(c("glmnet", "dplyr", "tidyr", "ggplot2"))

library(glmnet)   # cv.glmnet for LASSO (alpha=1) and Ridge (alpha=0)
library(dplyr)    # data manipulation
library(tidyr)    # reshaping
library(ggplot2)  # plotting

---
## STEP 1 — Load & Inspect Data

In [ ]:
load("data_ml.RData")

# ── Basic inspection ──────────────────────────────────────────────────────────
cat("Dataset dimensions :", nrow(data_ml), "rows x", ncol(data_ml), "cols\n")
cat("Date range          :", as.character(range(data_ml$date)), "\n")
cat("Unique stocks       :", length(unique(data_ml$stock_id)), "\n")
cat("Unique months       :", length(unique(data_ml$date)), "\n")
cat("\nAll column names:\n")
print(names(data_ml))

# ── Confirm the target column ─────────────────────────────────────────────────
# R1M_Usd = 'return forward 1 month' — already a forward return, no lagging needed
cat("\nSample of target column (R1M_Usd):\n")
print(summary(data_ml$R1M_Usd))

---
## STEP 2 — Preprocessing

Two operations applied **within each month** using only information available at that time:

1. **Winsorize** features at the 1st / 99th percentile (Paper 4 approach, prevents outlier-driven regularisation)
2. **Cross-sectionally standardise** (zero mean, unit SD) so the L1/L2 penalty treats all features equally

> **No lagging needed.** `R1M_Usd` is defined as the *forward* 1-month return, so features at time *t* already pair with the return earned from *t* to *t+1*. Applying `lead()` would incorrectly shift to *t+2*.
> The other label columns (`R3M_Usd`, `R6M_Usd`, `R12M_Usd`) are excluded from features to prevent leakage.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TARGET_COL  <- "R1M_Usd"                          # forward 1-month return (label)
ID_COLS     <- c("date", "stock_id")              # non-feature identifier columns
LABEL_COLS  <- c("R1M_Usd", "R3M_Usd",
                  "R6M_Usd", "R12M_Usd")          # all labels — exclude from features

FEATURE_COLS <- setdiff(names(data_ml), c(ID_COLS, LABEL_COLS))
cat("Number of features:", length(FEATURE_COLS), "\n")
cat("Features:\n")
print(FEATURE_COLS)

# ── Helper: winsorise a vector to [p_low, p_high] ────────────────────────────
winsorise <- function(x, p_low = 0.01, p_high = 0.99) {
  lo <- quantile(x, p_low,  na.rm = TRUE)
  hi <- quantile(x, p_high, na.rm = TRUE)
  pmin(pmax(x, lo), hi)
}

# ── Helper: cross-sectional standardise (within a single month's slice) ───────
cs_standardise <- function(x) {
  mu <- mean(x, na.rm = TRUE)
  sd <- sd(x,   na.rm = TRUE)
  if (is.na(sd) || sd == 0) return(x - mu)   # avoid divide-by-zero
  (x - mu) / sd
}

# ── Apply preprocessing month-by-month ───────────────────────────────────────
# R1M_Usd is already the forward 1-month return — no lead() needed.
# We only drop rows where the target itself is NA.
data_processed <- data_ml %>%
  arrange(date, stock_id) %>%
  filter(!is.na(R1M_Usd)) %>%                        # drop rows with no forward return
  group_by(date) %>%
  mutate(
    # Winsorise every feature cross-sectionally
    across(all_of(FEATURE_COLS), winsorise),
    # Standardise every feature cross-sectionally
    across(all_of(FEATURE_COLS), cs_standardise)
  ) %>%
  ungroup()

cat("\nPreprocessed dataset:\n")
cat("  Rows:", nrow(data_processed), "\n")
cat("  Feature NAs remaining:",
    sum(is.na(data_processed[, FEATURE_COLS])), "\n")

# ── Impute any remaining NAs with the cross-sectional mean (= 0 after standardising) ──
data_processed[, FEATURE_COLS][is.na(data_processed[, FEATURE_COLS])] <- 0

---
## STEP 3 — Rolling-Window Loop

### Design choices (document these in your report)
| Parameter | Value | Justification |
|-----------|-------|---------------|
| Window type | **Expanding** | Uses all available history; consistent with Papers 3 & 4 |
| Minimum train months | **60** (5 years) | Same as Paper 3's rolling window length |
| Lambda selection | **`lambda.1se`** | More parsimonious; preferred for factor investing (Eval_Metrics note) |
| CV folds | **5**, time-ordered | Avoids data leakage; last 20% of training window as validation |

### Output produced per month
- `stock_id`, `date`, `y_true` (actual forward 1M return = `R1M_Usd`), `y_pred_lasso`, `y_pred_ridge`
- `n_features_lasso` (how many non-zero coefs LASSO kept — for the path analysis)

In [ ]:
# ── Rolling-window configuration ──────────────────────────────────────────────
MIN_TRAIN_MONTHS <- 60   # minimum months before we start predicting
LAMBDA_RULE      <- "lambda.1se"   # or "lambda.min" — keep consistent
N_CV_FOLDS       <- 5

all_dates <- sort(unique(data_processed$date))
n_dates   <- length(all_dates)

# Pre-allocate results list (faster than rbind in a loop)
results_list        <- vector("list", n_dates - MIN_TRAIN_MONTHS)
lasso_feature_track <- vector("list", n_dates - MIN_TRAIN_MONTHS)  # tracks feature selection

cat("Starting rolling window. Predicting",
    n_dates - MIN_TRAIN_MONTHS, "months...\n")

# ── Main loop ─────────────────────────────────────────────────────────────────
for (i in seq(MIN_TRAIN_MONTHS, n_dates - 1)) {

  train_dates <- all_dates[1:i]          # EXPANDING window: all months up to i
  test_date   <- all_dates[i + 1]        # predict the very next month

  # ── Slice train / test ────────────────────────────────────────────────────
  train <- data_processed[data_processed$date %in% train_dates, ]
  test  <- data_processed[data_processed$date == test_date, ]

  X_train <- as.matrix(train[, FEATURE_COLS])
  y_train <- train$R1M_Usd
  X_test  <- as.matrix(test[, FEATURE_COLS])
  y_test  <- test$R1M_Usd

  # ── Time-series-safe CV: create fold IDs ordered chronologically ─────────
  # Split training rows into N_CV_FOLDS consecutive blocks (not random).
  # This ensures no future data leaks into any validation fold.
  n_train  <- nrow(X_train)
  fold_ids <- ceiling((seq_len(n_train) / n_train) * N_CV_FOLDS)
  fold_ids <- pmin(fold_ids, N_CV_FOLDS)   # clamp to [1, N_CV_FOLDS]

  # ── LASSO (alpha = 1) ─────────────────────────────────────────────────────
  set.seed(42)
  cv_lasso <- tryCatch(
    glmnet::cv.glmnet(
      X_train, y_train,
      alpha      = 1,             # L1 penalty → variable selection
      foldid     = fold_ids,      # time-ordered folds (no shuffle)
      standardize = FALSE         # already standardised cross-sectionally
    ),
    error = function(e) NULL
  )

  if (!is.null(cv_lasso)) {
    pred_lasso   <- as.numeric(predict(cv_lasso, X_test, s = LAMBDA_RULE))
    n_nonzero    <- sum(coef(cv_lasso, s = LAMBDA_RULE) != 0) - 1L  # excl. intercept
    lambda_lasso <- ifelse(LAMBDA_RULE == "lambda.1se",
                           cv_lasso$lambda.1se, cv_lasso$lambda.min)
  } else {
    pred_lasso   <- rep(NA_real_, nrow(X_test))
    n_nonzero    <- NA_integer_
    lambda_lasso <- NA_real_
  }

  # ── Ridge (alpha = 0) ─────────────────────────────────────────────────────
  set.seed(42)
  cv_ridge <- tryCatch(
    glmnet::cv.glmnet(
      X_train, y_train,
      alpha       = 0,            # L2 penalty → shrink all, select none
      foldid      = fold_ids,
      standardize = FALSE
    ),
    error = function(e) NULL
  )

  if (!is.null(cv_ridge)) {
    pred_ridge   <- as.numeric(predict(cv_ridge, X_test, s = LAMBDA_RULE))
    lambda_ridge <- ifelse(LAMBDA_RULE == "lambda.1se",
                           cv_ridge$lambda.1se, cv_ridge$lambda.min)
  } else {
    pred_ridge   <- rep(NA_real_, nrow(X_test))
    lambda_ridge <- NA_real_
  }

  # ── Store results for this month ──────────────────────────────────────────
  idx <- i - MIN_TRAIN_MONTHS + 1

  results_list[[idx]] <- data.frame(
    date         = test_date,
    stock_id     = test$stock_id,
    y_true       = y_test,
    y_pred_lasso = pred_lasso,
    y_pred_ridge = pred_ridge
  )

  lasso_feature_track[[idx]] <- data.frame(
    date          = test_date,
    n_nonzero     = n_nonzero,
    lambda_lasso  = lambda_lasso,
    lambda_ridge  = lambda_ridge
  )

  # Progress update every 12 months
  if (idx %% 12 == 0) cat("  Completed month", idx, "/ ",
                           n_dates - MIN_TRAIN_MONTHS, "\n")
}

cat("Rolling window complete.\n")

---
## STEP 4 — Collect & Inspect OOS Predictions

In [ ]:
# ── Combine all monthly results into single data frames ───────────────────────
oos_predictions  <- do.call(rbind, results_list)
feature_tracking <- do.call(rbind, lasso_feature_track)

cat("OOS predictions shape:", nrow(oos_predictions), "rows x",
    ncol(oos_predictions), "cols\n")
cat("OOS date range:", as.character(range(oos_predictions$date)), "\n")
cat("\nNAs in predictions:\n")
cat("  LASSO:", sum(is.na(oos_predictions$y_pred_lasso)), "\n")
cat("  Ridge:", sum(is.na(oos_predictions$y_pred_ridge)), "\n")

# ── Quick sanity check: correlation between predictions and actuals ────────────
cat("\nRaw Pearson IC (all months pooled):\n")
cat("  LASSO:", round(cor(oos_predictions$y_true,
                           oos_predictions$y_pred_lasso, use="complete.obs"), 4), "\n")
cat("  Ridge:", round(cor(oos_predictions$y_true,
                           oos_predictions$y_pred_ridge, use="complete.obs"), 4), "\n")

In [ ]:
# ── LASSO: number of non-zero features over time ──────────────────────────────
# This plot goes in your report — shows how sparsity evolves across market regimes.
plot(
  feature_tracking$date,
  feature_tracking$n_nonzero,
  type = "l", col = "steelblue", lwd = 1.5,
  xlab = "Date",
  ylab = "Non-zero LASSO coefficients",
  main = "LASSO Feature Selection Over the Rolling Window"
)
abline(h = median(feature_tracking$n_nonzero, na.rm = TRUE),
       col = "tomato", lty = 2)
legend("topright", legend = c("Non-zero count", "Median"),
       col = c("steelblue", "tomato"), lty = c(1, 2))

---
## STEP 5 — Build Portfolio Weights & Save Outputs

### What `Eval_Metrics.ipynb` needs from this notebook

| Eval_Metrics function | Variable it uses from here | Where it comes from |
|-----------------------|---------------------------|---------------------|
| `compute_mse()` | `y_true`, `y_pred_lasso` / `y_pred_ridge` | `oos_predictions` |
| `compute_rmse_mae()` | same | same |
| `compute_oos_r2()` | same | same |
| `compute_hit_ratio()` | same | same |
| `compute_ic()` | same | same |
| `compute_theils_u()` | same | same |
| `evaluate_lasso_path()` | `X_train_final`, `y_train_final` | last training window |
| `select_lasso_lambda()` | same | same |
| `summarise_lasso_selection()` | `cv_lasso_final`, `FEATURE_COLS` | last CV fit |
| `ridge_bias_variance()` | `fit_ridge_final`, `X_test_final`, `y_test_final` | last Ridge fit |
| `compute_sharpe()` | `ls_returns_lasso`, `ls_returns_ridge` | portfolio construction below |
| `compute_max_drawdown()` | same | same |
| `compute_turnover()` | `weights_lasso_mat`, `returns_mat` | portfolio construction below |

In [ ]:
# ── 5A: Build quintile long-short portfolio returns ───────────────────────────
# Each month: rank stocks by predicted return, go LONG top 20%, SHORT bottom 20%.
# Equal-weight within each quintile.

build_ls_portfolio <- function(pred_df, pred_col) {
  # pred_df   : data frame with columns date, y_true, and the prediction column
  # pred_col  : string name of the prediction column (e.g. "y_pred_lasso")
  
  pred_df %>%
    filter(!is.na(.data[[pred_col]])) %>%
    group_by(date) %>%
    mutate(
      quintile = ntile(.data[[pred_col]], 5)
    ) %>%
    summarise(
      ret_long  = mean(y_true[quintile == 5], na.rm = TRUE),   # top quintile
      ret_short = mean(y_true[quintile == 1], na.rm = TRUE),   # bottom quintile
      ret_ls    = ret_long - ret_short,                         # long-short spread
      n_long    = sum(quintile == 5),
      n_short   = sum(quintile == 1),
      .groups   = "drop"
    )
}

portfolio_lasso <- build_ls_portfolio(oos_predictions, "y_pred_lasso")
portfolio_ridge <- build_ls_portfolio(oos_predictions, "y_pred_ridge")

# Vectors of monthly long-short returns — passed directly to Eval_Metrics
ls_returns_lasso <- portfolio_lasso$ret_ls
ls_returns_ridge <- portfolio_ridge$ret_ls

cat("Portfolio summary (LASSO long-short):\n")
cat("  Months:", nrow(portfolio_lasso), "\n")
cat("  Mean monthly return:", round(mean(ls_returns_lasso, na.rm=TRUE), 4), "\n")
cat("  SD monthly return:  ", round(sd(ls_returns_lasso, na.rm=TRUE), 4), "\n")

In [ ]:
# ── 5B: Build weights matrix for compute_turnover() ──────────────────────────
# compute_turnover() in Eval_Metrics expects a T x N weights matrix.
# We build a simplified version: +1/n_long for long stocks, -1/n_short for short.

build_weights_matrix <- function(pred_df, pred_col) {
  # Wide weight matrix: rows = dates, cols = stock_ids, values = portfolio weight
  w_df <- pred_df %>%
    filter(!is.na(.data[[pred_col]])) %>%
    group_by(date) %>%
    mutate(
      quintile = ntile(.data[[pred_col]], 5),
      n_long   = sum(quintile == 5),
      n_short  = sum(quintile == 1),
      weight   = case_when(
        quintile == 5 ~  1 / n_long,
        quintile == 1 ~ -1 / n_short,
        TRUE           ~  0
      )
    ) %>%
    ungroup() %>%
    select(date, stock_id, weight)

  # Pivot to wide format
  w_wide <- w_df %>%
    tidyr::pivot_wider(names_from = stock_id, values_from = weight,
                       values_fill = 0) %>%
    arrange(date)

  as.matrix(w_wide[, -1])   # drop date column; return numeric matrix
}

weights_lasso_mat <- build_weights_matrix(oos_predictions, "y_pred_lasso")
weights_ridge_mat <- build_weights_matrix(oos_predictions, "y_pred_ridge")

cat("Weights matrix (LASSO) dimensions:", dim(weights_lasso_mat), "\n")

In [ ]:
# ── 5C: Build asset returns matrix for compute_turnover() ─────────────────────
# Same T x N shape as the weights matrix, aligned by date and stock_id.

returns_wide <- oos_predictions %>%
  select(date, stock_id, y_true) %>%
  tidyr::pivot_wider(names_from = stock_id, values_from = y_true,
                     values_fill = 0) %>%
  arrange(date)

returns_mat <- as.matrix(returns_wide[, -1])

# Align column order to weights matrix
common_cols  <- intersect(colnames(weights_lasso_mat), colnames(returns_mat))
weights_lasso_mat <- weights_lasso_mat[, common_cols]
weights_ridge_mat <- weights_ridge_mat[, common_cols]
returns_mat       <- returns_mat[, common_cols]

cat("Returns matrix dimensions:", dim(returns_mat), "\n")

In [ ]:
# ── 5D: Save the final training window — needed for path & CV plots ───────────
# Eval_Metrics functions evaluate_lasso_path(), select_lasso_lambda(),
# summarise_lasso_selection(), and ridge_bias_variance() all need a
# train/test split to plot on. We use the LAST rolling window for this.

last_train_dates  <- all_dates[1:(n_dates - 1)]
last_test_date    <- all_dates[n_dates]

last_train <- data_processed[data_processed$date %in% last_train_dates, ]
last_test  <- data_processed[data_processed$date == last_test_date, ]

X_train_final <- as.matrix(last_train[, FEATURE_COLS])
y_train_final <- last_train$R1M_Usd
X_test_final  <- as.matrix(last_test[, FEATURE_COLS])
y_test_final  <- last_test$R1M_Usd

# Fit the final models once (for the diagnostic plots)
set.seed(42)
fold_ids_final  <- ceiling((seq_len(nrow(X_train_final)) /
                            nrow(X_train_final)) * N_CV_FOLDS)
fold_ids_final  <- pmin(fold_ids_final, N_CV_FOLDS)

cv_lasso_final  <- glmnet::cv.glmnet(X_train_final, y_train_final,
                                      alpha = 1, foldid = fold_ids_final,
                                      standardize = FALSE)
fit_ridge_final <- glmnet::glmnet(X_train_final, y_train_final,
                                   alpha = 0, standardize = FALSE)

cat("Final-window models fit.\n")
cat("  LASSO lambda.1se :", round(cv_lasso_final$lambda.1se, 6), "\n")
cat("  Non-zero coefs   :",
    sum(coef(cv_lasso_final, s = "lambda.1se") != 0) - 1, "\n")

In [ ]:
# ── 5E: Save LASSO and Ridge outputs to SEPARATE .RData files ────────────────
# Each file is self-contained: it includes the shared matrices (X_train_final,
# X_test_final, y_train_final, y_test_final, returns_mat) so that either file
# can be loaded independently into Eval_Metrics.ipynb without needing the other.

# ── LASSO outputs → lasso_outputs.RData ──────────────────────────────────────
save(
  # Predictions (Parts A, G)
  oos_predictions,      # columns used: date, stock_id, y_true, y_pred_lasso

  # Portfolio (Part H)
  ls_returns_lasso,     # → compute_sharpe(), compute_max_drawdown()
  weights_lasso_mat,    # → compute_turnover()
  returns_mat,          # shared — needed by compute_turnover()

  # LASSO diagnostics (Part C)
  cv_lasso_final,       # → select_lasso_lambda(), summarise_lasso_selection()
  X_train_final,        # → evaluate_lasso_path()
  y_train_final,
  X_test_final,
  y_test_final,
  FEATURE_COLS,         # → summarise_lasso_selection() needs feature names

  # Feature selection over time (your report figure)
  feature_tracking,

  file = "lasso_outputs.RData"
)
cat("LASSO outputs saved to lasso_outputs.RData\n")

# ── Ridge outputs → ridge_outputs.RData ──────────────────────────────────────
save(
  # Predictions (Parts A, G)
  oos_predictions,      # columns used: date, permno, y_true, y_pred_ridge

  # Portfolio (Part H)
  ls_returns_ridge,     # → compute_sharpe(), compute_max_drawdown()
  weights_ridge_mat,    # → compute_turnover()
  returns_mat,          # shared — needed by compute_turnover()

  # Ridge diagnostics (Part D)
  fit_ridge_final,      # → ridge_bias_variance()
  X_train_final,        # shared — kept for symmetry / completeness
  y_train_final,
  X_test_final,
  y_test_final,
  FEATURE_COLS,

  file = "ridge_outputs.RData"
)
cat("Ridge  outputs saved to ridge_outputs.RData\n")
cat("\nLoad in Eval_Metrics.ipynb with:\n")
cat("  load('lasso_outputs.RData')   # for LASSO evaluation\n")
cat("  load('ridge_outputs.RData')   # for Ridge evaluation\n")

---
## How to Use the Outputs in `Eval_Metrics.ipynb`

Each model now has its own file. Load only what you need in `Eval_Metrics.ipynb`:

---

### LASSO evaluation — load `lasso_outputs.RData`

```r
load("lasso_outputs.RData")

y_true       <- oos_predictions$y_true
y_pred_lasso <- oos_predictions$y_pred_lasso

# PART A — shared regression metrics
compute_mse(y_true, y_pred_lasso)
compute_rmse_mae(y_true, y_pred_lasso, label = "LASSO")
compute_oos_r2(y_true, y_pred_lasso)
compute_hit_ratio(y_true, y_pred_lasso)

# PART C — LASSO-specific
evaluate_lasso_path(X_train_final, y_train_final)
select_lasso_lambda(X_train_final, y_train_final)
summarise_lasso_selection(cv_lasso_final, FEATURE_COLS)

# PART G — information coefficient
compute_ic(y_true, y_pred_lasso)
compute_theils_u(y_true, y_pred_lasso)

# PART H — portfolio
avg_to_lasso <- compute_turnover(weights_lasso_mat, returns_mat)
compute_sharpe(ls_returns_lasso,
               tc_per_unit_turnover = 0.005,
               avg_turnover         = avg_to_lasso)
compute_max_drawdown(ls_returns_lasso)
```

---

### Ridge evaluation — load `ridge_outputs.RData`

```r
load("ridge_outputs.RData")

y_true       <- oos_predictions$y_true
y_pred_ridge <- oos_predictions$y_pred_ridge

# PART A — shared regression metrics
compute_mse(y_true, y_pred_ridge)
compute_rmse_mae(y_true, y_pred_ridge, label = "Ridge")
compute_oos_r2(y_true, y_pred_ridge)
compute_hit_ratio(y_true, y_pred_ridge)

# PART D — Ridge-specific
ridge_bias_variance(fit_ridge_final, X_test_final, y_test_final)

# PART G — information coefficient
compute_ic(y_true, y_pred_ridge)
compute_theils_u(y_true, y_pred_ridge)

# PART H — portfolio
avg_to_ridge <- compute_turnover(weights_ridge_mat, returns_mat)
compute_sharpe(ls_returns_ridge,
               tc_per_unit_turnover = 0.005,
               avg_turnover         = avg_to_ridge)
compute_max_drawdown(ls_returns_ridge)
```